<div align="center">

# **LLM Visual Explorer V1**
## (PL, Aug26)

![LlmExpl — Visual Explorer](../images/LlmExpl_logo.png)

## Explorer l'espace d'un modèle de langage d'intelligence artificielle à travers un "trou de serrure"

</div>

Un mot/concept n'est pas un point isolé. Il possède une position dans un
espace sémantique de plusieurs centaines, voire milliers de dimensions.

---



# LLM Visual Explorer — Explorar la geometría del significado

**LLM Visual Explorer (LlmExpl)** es un pequeño laboratorio que permite explorar cómo diferentes modelos representan palabras y conceptos en espacios de cientos de dimensiones.

Actualmente ofrece **más de 30 escenarios**, en **francés, inglés y español**, que pueden explorarse con **4 modelos** diferentes.


## Ver el espacio semántico

Es imposible ver directamente cientos de dimensiones.

Por eso, LlmExpl utiliza mapas **2D** y un **planetario 3D**: una especie de «ojo de cerradura» que permite entrever esta geometría. Las **similitudes** permiten después medir las relaciones en el espacio original.


## Bajo el capó: sorpresas

La geometría obtenida **no está necesariamente organizada según nuestra intuición humana**.

Depende del modelo, del idioma y de las palabras elegidas: conceptos que intuitivamente parecen cercanos pueden alejarse, mientras que pueden aparecer relaciones sorprendentes.


## ¿Y antes del significado?

El modelo no recibe directamente «conceptos», sino texto dividido en **tokens**.

Terminaremos observando la tokenización y, después, cómo el **contexto** puede modificar la representación de palabras ambiguas como *avocat*, *mouse* o *banco*.

> **Ver → explorar → comparar → medir → cuestionar**

In [ ]:
#=======================================================
# 0 — Initialization and required imports
#=======================================================

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from explorer.notebook_setup import *

In [ ]:
# Test model pulldown

import ipywidgets as widgets
from IPython.display import display
import explorer.config as config


model_selector = widgets.Dropdown(
    options=[
        ("MiniLM — fast & compact", "MiniLM"),
        ("MPNet — semantic representation", "MPNet"),
        ("BERT — historical reference", "BERT"),
        ("LaBSE — multilingual", "LaBSE"),
    ],
    value=config.ACTIVE_MODEL,
    description="Model:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="430px"),
)


def update_active_model(change):
    if change["name"] == "value":
        config.ACTIVE_MODEL = change["new"]
        print(f"Active model: {config.ACTIVE_MODEL}")


model_selector.observe(
    update_active_model,
    names="value",
)

display(model_selector)

In [ ]:
#=================================================
# 1 — Scenario selection    
#=================================================

from explorer.scenarios import scenario_selector
from explorer.display import display_scenario

selection = {}

scenario_selector(
    display_function=display_scenario,
    selection=selection
)

In [ ]:
# ==========================================================
# 2 — Model loading and semantic vector computation
# ==========================================================

# Retrieve the selected scenario data
scenario = selection["scenario"]
concepts = selection["concepts"]
objects = selection["objects"]
icons = selection["icons"]
short_names = selection.get("short_names", {})

# Load the model and compute embeddings / semantic vectors
embeddings = compute_embeddings(concepts)

# Retrieve the loaded model
model = get_model()

# Display model information and embedding dimension
display_model(model, concept_count=len(concepts))

In [ ]:
#================================================
# 3 — Projection from model space to 3D and DataFrame creation
#================================================
# Compute the projection
xyz, pca = compute_pca( embeddings )

# Display projection quality / fidelity (maximum = 100%)
display_projection( pca )

# Prepare data for visualization
df = create_dataframe( concepts, xyz )

In [ ]:
#================================================
# 4 — 2D Semantic Map
#================================================

# This map projects concepts into a mathematical space.
# Nearby points represent concepts with similar vector representations.

fig_map = plot_map(
    df=df,
    title=tr("semantic_map_2d"),
    icons=icons,
    short_names=short_names,
    show_labels=True,
    show_icons=False,
)

fig_map.show()

In [ ]:
#==============================================
# 5 — 3D Semantic Planetarium
#==============================================

fig = plot_scene(
    df=df,
    title=tr("semantic_planetarium_3d"),
    icons=icons,
    short_names=short_names,
    show_labels=True,
    show_icons=False,
)

fig.show()

In [ ]:
#================================================
# 6 — Similarity Ranking
#================================================

from explorer.exports import save_similarities_csv

similarities = compute_similarity(embeddings)

similarity_pairs = rank_similarity_pairs(
    concepts,
    similarities
)

display_similarity_ranking(
    similarity_pairs,
    df,
    icons,
    top_n=20,
    bottom_n=10,
)

save_similarities_csv(
    similarity_pairs,
    scenario_name=selection["scenario_name"],
    model_alias=config.get_model_config()["alias"],
    model_name=config.get_model_config()["name"],
)


In [ ]:
# ============================================================
# 7 — Progressive Semantic Core Construction
# ============================================================

core_order, semantic_strength = build_semantic_core(
    embeddings,
    concepts,
    strength_mode=config.PLANETARIUM_STRENGTH_MODE,
)

display_semantic_core(
    core_order,
    semantic_strength,
    strength_mode=config.PLANETARIUM_STRENGTH_MODE,
)

In [ ]:
# ============================================================
# 8 — Semantic-Strength Planetarium
# ============================================================

fig = plot_scene(
    df,
    title=tr("semantic_strength_planetarium"),
    icons=icons,
    short_names=short_names,
    semantic_strength=semantic_strength,
    strength_mode=config.PLANETARIUM_STRENGTH_MODE,
)
fig.show()

Before a language model processes text, it does not necessarily read whole words.
The text is first split into tokens: words, parts of words, punctuation marks, or other units depending on the tokenizer used by the model.
Let's see for example how different models "tokenize" a **word like crocodile and others**

**A word that looks simple to us may therefore be represented by one token… or several.**

In [ ]:
#=================================================
# 9 — Tokenization
#=================================================

import explorer.config as config

model_name = config.get_model_config()["name"]

tokenizer = AutoTokenizer.from_pretrained(model_name)

print(model_name)

**Context & Polysemy - LLM Layers**

Here we explore the three polysemic (ambiguous) words **avocat(fr), mouse(en) and banco(sp)**. The word itself remains the same. 
Only its context changes. As the word passes through the Transformer layers, its vector representation is transformed together with the representations of the other tokens in the sentence.

We follow this representation layer by layer and compare it with contextual semantic prototypes built from the same word used in sentences corresponding to each possible meaning.

**We do not directly observe “meaning”. We observe how an internal representation evolves, and probe it through semantic similarity.**

In [ ]:
#=================================================
# 10 — Context & Polysemy
#=================================================

from explorer.context import context_explorer

context_explorer()

**What to notice**

The same word follows different trajectories depending on its context.

Across the Transformer layers, its representation becomes increasingly aligned with the contextual prototype corresponding to the meaning suggested by the sentence.

In [ ]:
#======================================================================
# 11 — Exploration Report
#======================================================================

display_report(
    scenario=scenario,
    concepts=concepts,
    embedding_dimension=embeddings.shape[1],
    pca=pca,
    similarity_pairs=similarity_pairs,
    planetarium_figure=fig,
)


# Conclusión — Algunas ideas para recordar

## 1. Las palabras —o «conceptos»— tienen una geometría

No están simplemente almacenadas en un diccionario. Los modelos las representan en un **espacio semántico** de cientos, 
o incluso miles, de dimensiones.

Con **LLM Visual Explorer**, podemos intentar «ver» este espacio.


## 2. Pero lo observamos a través de un ojo de cerradura

Un espacio de cientos de dimensiones no puede representarse directamente en una pantalla. Los mapas 2D y los planetarios 3D son **proyecciones**: conservan una parte de la estructura, pero necesariamente deforman o pierden otra.

Por eso, la visualización nos permite explorar, mientras que las **similitudes calculadas en el espacio original** nos permiten medir.


## 3. La proximidad semántica es estadística, no lógica

Dos conceptos cercanos no son necesariamente sinónimos. Su proximidad refleja patrones aprendidos por el modelo a partir de los **milliones de textos con los que fue entrenado.**

Por eso, una relación puede parecernos intuitiva… o, a veces, muy sorprendente.


## 4. Los modelos aprenden nuestro lenguaje…

…pero no necesariamente **nuestra visión del mundo**.

A partir de enormes cantidades de texto, estos sistemas aprenden patrones estadísticos de los que pueden **emerger estructuras y comportamientos sorprendentemente complejos**.

Estas representaciones dependen del **modelo**, del **idioma**, de la **tokenización** y del **contexto**.


---

### LLM Visual Explorer ofrece así un pequeño laboratorio para:

**ver → explorar → comparar → medir → cuestionar**

> **¿Qué vemos realmente cuando dibujamos un mapa del «significado» de un modelo?**

In [ ]:
# ================================================================
# 8 — Messages: Exploration Conclusion
# ================================================================

from explorer.display import display_messages

display_messages()

In [ ]:
#===================================================
# Test Cell
#===================================================

